In [1]:
import torch 
import torch.nn as nn 
import matplotlib.pyplot as plt

In [3]:
z = torch.linspace(0,2,100).reshape(-1,1)
z.requires_grad_(True)
omega_m = 0.3
class ΛCDM_NN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(1,19)
        self.tanh = nn.Tanh()
        self.layer2 = nn.Linear(19,1)

    def forward(self,z):
        z = self.layer1(z)
        z = self.tanh(z)
        z = self.layer2(z)
        return z
model = ΛCDM_NN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
H_true = torch.sqrt(omega_m * (1 + z)**3 + (1-omega_m)).detach()

for epoch in range(1000):
    H_pred = model(z)
    loss_data = torch.mean((H_pred - H_true)**2)
    optimizer.zero_grad()
    loss_data.backward()
    optimizer.step()
    if epoch %100 ==0:
        print(epoch, loss_data.item())

0 2.8408725261688232
100 0.017968326807022095
200 0.008797998540103436
300 0.0045376792550086975
400 0.0022428464144468307
500 0.0009705178672447801
600 0.00037307056481949985
700 0.00015097801224328578
800 8.174247341230512e-05
900 6.000902794767171e-05


In [5]:
z = torch.linspace(0,2,1000).reshape(-1,1)
z.requires_grad_(True)
omega_m = 0.3
class ΛCDM_pinn(nn.Module):
    def __init__(self):
        super().__init__()
        self.omega_m2 = nn.Parameter(torch.tensor(0.5))
        self.layer1 = nn.Linear(1,19)
        self.tanh = nn.Tanh()
        self.layer2 = nn.Linear(19,1)

    def forward(self,z):
        z = self.layer1(z)
        z = self.tanh(z)
        z = self.layer2(z)
        return z

model = ΛCDM_pinn()
optimizer = torch.optim.Adam(model.parameters(), lr =0.001)
z0 = torch.tensor([[0.0]])
H0 = torch.tensor([[1.0]]) 
H_true = torch.sqrt(omega_m * (1 + z)**3 + (1-omega_m)).detach()


for epoch in range(1000):
    H_pred = model(z)
    dH_dz = torch.autograd.grad(
        outputs = H_pred,
        inputs = z,
        create_graph=True,
        grad_outputs=torch.ones_like(H_pred))[0]
    residual = dH_dz - 3 * model.omega_m2 * (1+z)**2 / (2 * H_pred)
    loss_r = torch.mean(residual**2)
    H0_pred = model(z0)
    loss_ic = torch.mean((H0_pred-H0)**2)
    loss_data = torch.mean((H_pred - H_true)**2)
    loss = loss_r + loss_ic + loss_data
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch %100 == 0:
         print(
        epoch,
        "total =", loss.item(),
        "L_r =", loss_r.item(),
        "L_ic =", loss_ic.item(),
        "L_data =", loss_data.item())
    
    
        

0 total = 30.368194580078125 L_r = 28.300281524658203 L_ic = 0.3307827413082123 L_data = 1.737130880355835
100 total = 3.6193714141845703 L_r = 3.2348580360412598 L_ic = 0.022489551454782486 L_data = 0.3620239198207855
200 total = 1.4623914957046509 L_r = 1.3449938297271729 L_ic = 0.002809262601658702 L_data = 0.11458836495876312
300 total = 0.747202455997467 L_r = 0.703685462474823 L_ic = 0.000701295561157167 L_data = 0.042815666645765305
400 total = 0.47312819957733154 L_r = 0.44370681047439575 L_ic = 0.0012390651972964406 L_data = 0.028182314708828926
500 total = 0.360181599855423 L_r = 0.3293215334415436 L_ic = 0.0034317190293222666 L_data = 0.027428366243839264
600 total = 0.3059074282646179 L_r = 0.27214711904525757 L_ic = 0.006903473753482103 L_data = 0.026856839656829834
700 total = 0.27253180742263794 L_r = 0.23759686946868896 L_ic = 0.010320983827114105 L_data = 0.024613935500383377
800 total = 0.24694523215293884 L_r = 0.21268440783023834 L_ic = 0.012557766400277615 L_data =

In [6]:
model.omega_m2.item()

0.2995717227458954